# 07 — Taiwan Biobank external validation

The pooled model is applied to a cohort it has never seen. Taiwan Biobank
measures no fasting insulin, so HOMA-IR — and therefore the observed `IR` label —
cannot be computed for it. **There is no ground truth here and no AUC.** What
this notebook produces instead is a *predicted* prevalence, a characterisation of
the predicted-positive group, and two distributional comparisons that ask whether
the predictions behave like real labels.

It also writes `TWB_with_IR.parquet`, which notebook 08 consumes: the methylation
subset is the 1,199 participants carrying a `MET_ID`, split by this predicted
label.

**Which model does the scoring.** The top-20 CatBoost, not the 241-feature one.
The choice is material — the two differ by 381 participants on this cohort
(19,596 against 19,977) — so it is stated rather than left to be inferred.

**Run order.** This notebook appends a fourth sheet to the workbooks that
notebook 02 creates, so 02 must have been run first.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pickle

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import ks_2samp

from src.data.io import output_path, processed_path, repo_root, write_parquet
from src.logging_utils import configure_logging
from src.models.evaluate import cohens_d, predict_labels
from src.viz.figures import (
    plot_feature_boxplots,
    plot_quantile_comparison,
    plot_tg_hdl_boxplots,
)
from src.viz.tables import descriptive_statistics

configure_logging(ROOT / "logs")

nhanes = pl.read_parquet(processed_path("NHANES_data.parquet"))
knhanes = pl.read_parquet(processed_path("KNHANES_data.parquet"))
twb_clinical = pl.read_parquet(processed_path("TWB_clinical_data.parquet"))
twb_features = pl.read_parquet(processed_path("TWB_race2_features.parquet"))

with open(repo_root() / "models" / "catboost_top20.pkl", "rb") as handle:
    model = pickle.load(handle)

print(f"TWB features {twb_features.shape}, model expects {len(model.feature_names_)} features")

TWB features (92734, 243), model expects 20 features


## Scoring

The label is added as an integer rather than a boolean: it is a prediction, and
keeping it visibly different in type from the measured `IR` of the other two
cohorts makes it harder to conflate the two downstream.

In [2]:
labels = predict_labels(model, twb_features)

twb = twb_features.with_columns(IR=pl.Series(labels))
twb = twb.select(sorted(twb.columns))
write_parquet(twb, processed_path("TWB_with_IR.parquet"))

positive = int(labels.sum())
print(f"{twb.height:,} participants scored")
print(f"  IR+ {positive:,} ({positive / twb.height:.1%})")
print(f"  IR- {twb.height - positive:,}")

methylation = twb.filter(pl.col("MET_ID") != "")
methylation_positive = methylation.filter(pl.col("IR") == 1).height
print()
print(f"methylation subset: {methylation.height:,} participants")
print(f"  IR+ {methylation_positive:,}, IR- {methylation.height - methylation_positive:,}")

2026-09-16 23:31:07 [INFO] src.models.evaluate: Predicted 19596 of 92734 rows positive (21.1%)


92,734 participants scored
  IR+ 19,596 (21.1%)
  IR- 73,138

methylation subset: 1,199 participants
  IR+ 246, IR- 953


## Cohort characteristics of the predicted groups

The same table as notebook 02, over the clinical variables, split by the
*predicted* label. The column order and the `mean±SD` formatting are identical to
the other three sheets; only the meaning of the split differs, which is why the
sheet is named `TWB(predict)`.

Taiwan Biobank has no `FASTING_INSULIN` and no `HOMA-IR`, so those two rows are
absent here. The sheet is appended to both workbooks — thresholded p-values in
`stats.xlsx`, raw p-values in `stats_original.xlsx`.

In [3]:
assert twb["Release_No"].to_list() == twb_clinical["Release_No"].to_list(), (
    "row alignment between the clinical table and the scored table broke"
)
twb_labelled = pl.concat([twb_clinical, twb.select("IR") == 1], how="horizontal")

twb_table = descriptive_statistics(twb_labelled)
twb_table_raw = descriptive_statistics(twb_labelled, alpha=None)

for filename, table in [
    ("stats.xlsx", twb_table),
    ("stats_original.xlsx", twb_table_raw),
]:
    with pd.ExcelWriter(
        output_path(filename), mode="a", engine="openpyxl", if_sheet_exists="replace"
    ) as writer:
        table.to_excel(writer, sheet_name="TWB(predict)", index=False)

twb_table


2026-09-16 23:31:09 [INFO] src.viz.tables: Descriptive statistics table: 18 rows


2026-09-16 23:31:09 [INFO] src.viz.tables: Descriptive statistics table: 18 rows


,column,All,IR-,IR+,ks_p_value-,ks_p_value+,t_test_p_value,u_test_p_value
0,N,"92,734","73,138","19,596",None,None,None,None
1,Male,"34,859(37.6%)","24,393(33.4%)","10,466(53.4%)",None,None,None,None
2,Female,"57,875(62.4%)","48,745(66.6%)","9,130(46.6%)",None,None,None,None
3,AGE,46.4±13.2,46.2±13.1,47.0±13.7,< 0.01,< 0.01,< 0.01,< 0.01
4,BMI,23.9±3.9,22.7±2.9,28.5±3.8,< 0.01,< 0.01,< 0.01,< 0.01
5,BODY_WAISTLINE,82.1±10.8,79.0±8.5,93.8±10.6,< 0.01,< 0.01,< 0.01,< 0.01
6,BUN,12.8±3.7,12.7±3.6,13.2±3.8,< 0.01,< 0.01,< 0.01,< 0.01
7,CREATININE,0.7±0.3,0.7±0.3,0.8±0.3,< 0.01,< 0.01,< 0.01,< 0.01
8,FASTING_GLUCOSE,93.0±13.1,90.2±6.8,103.4±22.3,< 0.01,< 0.01,< 0.01,< 0.01
9,HBA1C,5.6±0.5,5.5±0.3,6.0±0.8,< 0.01,< 0.01,< 0.01,< 0.01


## Distributions beside the labelled cohorts

The notebook 02 boxplot grid with a third cohort added. If the predictions were
noise, the TWB IR+ group would look like the TWB IR− group; instead it separates
on the same variables, in the same direction, as the two measured cohorts.

Two things about this figure are worth stating plainly:

- The implausible-value filter (`BODY_WAISTLINE < 500`, `T_CHO < 1000`) is
  applied **to Taiwan Biobank only**. Each condition removes exactly one row.
- The statistics table above applies no such filter, so it is computed over all
  92,734 participants while this figure uses 92,732.

In [4]:
BOXPLOT_EXCLUDED = [
    "Release_No", "DIABETES", "SEX", "MET_ID", "FASTING_INSULIN", "HOMA-IR"
]

labelled_panels = pl.concat(
    [
        nhanes.with_columns(RACE=pl.lit("NHANES")),
        knhanes.with_columns(RACE=pl.lit("KNHANES")),
    ]
).select(pl.all().exclude(BOXPLOT_EXCLUDED))
labelled_panels = labelled_panels.select(sorted(labelled_panels.columns))

twb_panel = twb_labelled.select(pl.all().exclude(BOXPLOT_EXCLUDED)).with_columns(
    RACE=pl.lit("TWB\n(IR was predicted)")
)
twb_panel = twb_panel.select(sorted(twb_panel.columns))
twb_panel = twb_panel.filter(pl.col("BODY_WAISTLINE") < 500)
twb_panel = twb_panel.filter(pl.col("T_CHO") < 1000)
twb_panel = twb_panel.with_columns(pl.col(pl.Int64).cast(pl.Float64))

print(f"TWB rows dropped by the implausible-value filter: {twb_labelled.height - twb_panel.height}")

plot_feature_boxplots(
    pl.concat([labelled_panels, twb_panel]),
    "Boxplots of variables by IR and Race (TWB was included)",
    output_path("boxplot_of_variable_by_IR_and_Race(Add TWB).png"),
);


TWB rows dropped by the implausible-value filter: 2


2026-09-16 23:31:13 [INFO] src.viz.figures: Wrote output/boxplot_of_variable_by_IR_and_Race(Add TWB).png


## The TG/HDL-C ratio

The triglyceride to HDL-cholesterol ratio is an established surrogate marker of
insulin resistance that the model was never told about — `TG` and `HDL_C` enter
as separate features. If the predicted labels are meaningful, the TWB panel
should separate on this ratio the way the measured cohorts do.

The plotting function takes a mapping of panel title to cohort, so a panel cannot
end up showing a different cohort from the one its title names.

In [5]:
plot_tg_hdl_boxplots(
    {
        "NHANES": nhanes,
        "KNHANES": knhanes,
        "NHANES + KNHANES": pl.concat([nhanes, knhanes]),
        "TWB (IR was predicted)": twb,
    },
    output_path("boxplot_of_TG_HDL_C_by_IR_and_Datasets.png"),
);


2026-09-16 23:31:13 [INFO] src.viz.figures: Wrote output/boxplot_of_TG_HDL_C_by_IR_and_Datasets.png


## How far apart are the two positive groups?

Restricted to insulin-resistant participants, how does the TG/HDL-C ratio of the
*measured* positives compare with that of the *predicted* positives? The Q-Q plot
answers it across the whole distribution; Cohen's *d* reduces it to one number.

The Q-Q plot uses the log ratio, because quantiles of a right-skewed variable are
easier to read on a log scale; the KS test and Cohen's *d* use the raw ratio, so
that the effect size is on the scale the ratio is normally quoted in. Both are
printed at full precision below.

In [6]:
RATIO = pl.col("TG") / pl.col("HDL_C")

labelled_positive = pl.concat([nhanes, knhanes]).filter(pl.col("IR"))
twb_positive = twb.filter(pl.col("IR") == 1)


def ratio_values(frame: pl.DataFrame, log: bool) -> np.ndarray:
    """Extract TG/HDL-C for one cohort, optionally on a log scale."""
    expression = np.log(RATIO) if log else RATIO
    return frame.select(expression.alias("value"))["value"].to_numpy()


plot_quantile_comparison(
    ratio_values(labelled_positive, log=True),
    ratio_values(twb_positive, log=True),
    ("NHANES+KNHANES", "TWB"),
    "QQ-Plot of log(TG/HDL-C) : NHANES+KNHANES vs TWB (only IR+)",
    output_path("qqplot_NHANES+KNHANES_vs_TWB.png"),
)

reference_raw = ratio_values(labelled_positive, log=False)
twb_raw = ratio_values(twb_positive, log=False)

ks_result = ks_2samp(reference_raw, twb_raw)
effect_size = cohens_d(reference_raw, twb_raw)

print(f"measured IR+ {len(reference_raw):,}, predicted IR+ {len(twb_raw):,}")
print(f"KS statistic {ks_result.statistic!r}")
print(f"KS p-value   {ks_result.pvalue!r}")
print(f"Cohen's d    {effect_size!r}")

2026-09-16 23:31:14 [INFO] src.viz.figures: Wrote output/qqplot_NHANES+KNHANES_vs_TWB.png


measured IR+ 9,409, predicted IR+ 19,596
KS statistic 0.07754907175752629
KS p-value   1.0689005058246168e-33
Cohen's d    -0.1332737447580242


The KS test rejects equality of the two distributions, but on samples this large
it would reject almost any difference. Cohen's *d* of −0.13 is what carries the
interpretation: a negligible-to-small effect, with the predicted-positive group
sitting slightly *higher* on TG/HDL-C than the measured one.
